# Getting Started: Testing an AVM Module Upgrade

**Scenario**: Test the upgrade of the Azure Application Gateway AVM module
from the **azurerm-based** version (`main`) to the **AzAPI-based** version (`feat/azapi-migration`).

This notebook walks through the full pipeline the testing agent follows:
1. **Discovery** -- scan the module to understand its structure
2. **Deploy (old version)** -- deploy the `main` branch default example
3. **Plan the upgrade** -- switch to the AzAPI branch and capture a structured plan diff
4. **Idempotency check** -- apply the new version and verify no drift
5. **Analysis** -- cross-reference plan changes against UPGRADE.md
6. **Report** -- summarise findings

> **Prerequisites**: Python 3.12+, Terraform CLI, Azure CLI (logged in), `gh` CLI, an Azure subscription for test resources.

## 1. Setup

Install the agent dependencies and configure the environment.

In [ ]:
# Install agent dependencies (run once)
!pip install -q -r requirements.txt

In [ ]:
import os, json, sys, subprocess
from pathlib import Path

# Point to the module under test
# Set MODULE_PATH env var or edit the default below for your environment
MODULE_PATH = os.environ.get(
    "MODULE_PATH",
    os.path.expanduser("~/src/modules/terraform-azurerm-avm-res-network-applicationgateway"),
)
if not Path(MODULE_PATH).exists():
    print(f"WARNING: Module not found at {MODULE_PATH}")
    print("Set the MODULE_PATH environment variable to your module location")

# Ensure .env is loaded
from dotenv import load_dotenv
load_dotenv(override=True)

# Import agent config
from config import config
print(f"Project endpoint: {config.project_endpoint[:40]}..." if config.project_endpoint else "WARNING: AZURE_AI_PROJECT_ENDPOINT not set")
print(f"Target subscription: {config.default_subscription_id or "(not set)"}")
print(f"Test RG prefix: {config.test_rg_prefix}")
print(f"Cleanup on complete: {config.cleanup_on_complete}")

## 2. Discovery -- Scan the Module

The Discovery agent's job is to ingest the module and build a `ModuleMap` --
a structured summary of everything available (examples, skills, tests,
UPGRADE.md, provider requirements).

In multi-agent mode this runs as a dedicated agent; here we call the
tools directly to see what they do.

In [ ]:
from tools.module_discovery import (
    ingest_local_module, discover_module_structure,
    read_module_skill, list_module_examples,
)

# Step 1: Ingest the module into an agent workspace
ingest_result = json.loads(ingest_local_module(MODULE_PATH))
workspace = ingest_result["workspace"]
print(f"Workspace: {workspace}")
print(f"Module copied to: {ingest_result['path']}")

In [ ]:
# Step 2: Discover the module structure
structure = json.loads(discover_module_structure(workspace))
print(json.dumps(structure, indent=2))

In [ ]:
# Step 3: List examples available for testing
examples = json.loads(list_module_examples(workspace))
print(f"Found {len(examples['examples'])} examples:")
for ex in examples["examples"]:
    print(f"  - {ex['name']} ({len(ex.get('files', []))} files)")

In [ ]:
# Step 4: Read the example-test skill (the MUT's own testing workflow)
skill = json.loads(read_module_skill(
    workspace, "AVM-Terraform-Development/references/example-test.md"
))
if skill["status"] == "success":
    print(skill["content"][:1000])
    print("... (truncated)")
else:
    print(f"Skill not found: {skill}")

In [ ]:
# Step 5: Read UPGRADE.md (critical for upgrade testing)
from tools.analysis import read_upgrade_doc

upgrade = json.loads(read_upgrade_doc(workspace))
if upgrade["status"] == "found":
    print(f"UPGRADE.md: {len(upgrade['content'])} chars")
    for line in upgrade["content"].split("\n")[:30]:
        print(line)
else:
    print("No UPGRADE.md found -- this is a finding if breaking changes exist")

## 3. Understanding the Upgrade

Before deploying, let's understand what changed between versions.

The Application Gateway module migration from azurerm to AzAPI involves:
- **Provider change**: core resource moves from `azurerm_application_gateway` to `azapi_resource`
- **Variable shape change**: `map(object)` with flat fields --> `list(object)` with nested `properties`
- **Cross-references**: name-based (`probe_name = "my-probe"`) --> ARM ID-based (`probe = { id = "..." }`)
- **Public IP**: no longer managed by the module (external resource)
- **New requirement**: Terraform >= 1.12, AzAPI >= 2.7

In [ ]:
# Examine provider requirements on each branch
def git_show(ref, path):
    r = subprocess.run(
        ["git", "-C", MODULE_PATH, "show", f"{ref}:{path}"],
        capture_output=True, text=True,
    )
    return r.stdout if r.returncode == 0 else f"(not found: {r.stderr.strip()})"

print("=== main branch (azurerm) ===")
print(git_show("main", "terraform.tf"))
print("\n=== feat/azapi-migration branch (AzAPI) ===")
print(git_show("feat/azapi-migration", "terraform.tf"))

## 4. Deploy the Old Version (main branch)

We deploy the `default` example from the `main` branch first.
This establishes the baseline state in Azure.

> **Note**: This creates real Azure resources. Ensure you have a valid subscription
> and permissions. The agent uses resource groups prefixed with `rg-avm-test-`
> and tags them for easy cleanup.

In [ ]:
from tools.terraform import (
    create_workspace, terraform_init, terraform_init_upgrade,
    terraform_plan_json, terraform_apply, check_idempotency,
    terraform_destroy, write_workspace_file, read_workspace_file,
    list_workspace_files,
)
from tools.azure import create_resource_group, delete_resource_group

# Create a fresh workspace for the old version
old_ws_result = json.loads(create_workspace("appgw-old"))
old_ws = old_ws_result["workspace"]
print(f"Old version workspace: {old_ws}")

### Deploy lifecycle

The deploy agent follows the workflow from the MUT's `example-test.md` skill:

```
terraform init -upgrade
    --> terraform plan -out=tfplan
        --> terraform apply tfplan
            --> terraform plan (idempotency -- should be empty)
                --> terraform destroy (cleanup)
```

Each step returns structured JSON. Raw terraform output stays inside the
deploy agent and is never passed to other agents.

In [ ]:
# Step 1: Init
# init_result = json.loads(terraform_init(old_ws))
# print(f"Init: {init_result['status']}")

# Step 2: Plan (structured JSON output)
# plan_result = json.loads(terraform_plan_json(old_ws))
# print(f"Plan: creates={plan_result['summary']['creates']}, "
#       f"updates={plan_result['summary']['updates']}, "
#       f"deletes={plan_result['summary']['deletes']}")

# Step 3: Apply
# apply_result = json.loads(terraform_apply(old_ws))
# print(f"Apply: {apply_result['status']}")

# Step 4: Idempotency check
# idem_result = json.loads(check_idempotency(old_ws))
# print(f"Idempotency: {idem_result['status']} "
#       f"(unexpected changes: {idem_result['unexpected_changes']})")

print("(Deploy steps commented out -- uncomment to run against real Azure)")
print("In production, the Deploy agent runs these automatically.")

## 5. Plan the Upgrade (AzAPI branch)

Now we switch the module source to the AzAPI version and run
`terraform init -upgrade` + `terraform plan -json` to capture
a structured diff of what would change.

This is the core of upgrade testing: the plan diff tells us exactly
which resources would be replaced, updated, or destroyed.

In [ ]:
# Simulated plan summary for the azurerm -> AzAPI migration
from models import PlanSummary, DeployResult, IdempotencyResult

upgrade_plan = PlanSummary(
    creates=15,   # New AzAPI resources
    updates=2,    # Modified supporting resources
    deletes=12,   # Old azurerm resources removed
    replaces=1,   # Application gateway itself (delete+create)
    no_ops=5,     # Unchanged resources (naming, regions, etc.)
)

print("Upgrade plan summary:")
print(f"  Creates:  {upgrade_plan.creates}")
print(f"  Updates:  {upgrade_plan.updates}")
print(f"  Deletes:  {upgrade_plan.deletes}")
print(f"  Replaces: {upgrade_plan.replaces} <-- BREAKING: resource replacement")
print(f"  No-ops:   {upgrade_plan.no_ops}")
print(f"  Total changes: {upgrade_plan.total_changes}")

## 6. Analysis -- Cross-reference with UPGRADE.md

The Analysis agent receives the structured plan summary and
cross-references it against the module's UPGRADE.md documentation.

It checks:
- Are the observed breaking changes documented?
- Are there documented changes that weren't triggered by the test?
- Are variable mappings accurate?

In [ ]:
from models import AnalysisFinding

# Simulated analysis findings for the app gateway upgrade
findings = [
    AnalysisFinding(
        category="breaking_change",
        severity="critical",
        description=(
            "Application gateway resource is replaced (delete+create). "
            "This causes downtime. The azurerm_application_gateway resource "
            "is destroyed and recreated as an azapi_resource."
        ),
        evidence={
            "resource": "module.application_gateway.azurerm_application_gateway.this",
            "action": "delete",
            "replacement": "module.application_gateway.azapi_resource.this",
        },
        upgrade_md_reference="Breaking changes summary",
    ),
    AnalysisFinding(
        category="breaking_change",
        severity="warning",
        description=(
            "Variable 'resource_group_name' removed, replaced by 'parent_id' "
            "which takes a full ARM resource ID instead of just the name."
        ),
        evidence={
            "old_var": "resource_group_name",
            "new_var": "parent_id",
            "type_change": "string (name) -> string (ARM ID)",
        },
        upgrade_md_reference="Variable mapping table",
    ),
    AnalysisFinding(
        category="breaking_change",
        severity="warning",
        description=(
            "Backend address pools changed from map(object) with flat fields "
            "to list(object) with nested 'properties' block matching ARM schema."
        ),
        evidence={
            "old_shape": "map(object({name, ip_addresses, fqdns}))",
            "new_shape": "list(object({name, properties = {backend_addresses = [{ip_address}]}}))",
        },
        upgrade_md_reference="Variable mapping table",
    ),
    AnalysisFinding(
        category="suggestion",
        severity="info",
        description=(
            "Public IP is no longer managed by the module. Users must create "
            "and manage their public IP externally. This should be highlighted "
            "prominently in migration guides."
        ),
        evidence={
            "removed_vars": ["create_public_ip", "public_ip_name", "public_ip_resource_id"],
        },
        upgrade_md_reference="Breaking changes summary",
    ),
]

print(f"Analysis produced {len(findings)} findings:\n")
for i, f in enumerate(findings, 1):
    icon = {"critical": "X", "warning": "!", "info": "i"}[f.severity]
    print(f"  {i}. [{icon}] [{f.category}] {f.description[:80]}...")
    if f.upgrade_md_reference:
        print(f"     UPGRADE.md ref: {f.upgrade_md_reference}")
    print()

## 7. Review -- Cross-check Findings

The Reviewer agent gets a fresh context (no terraform output fog) and
validates each finding independently. This catches false positives.

In [ ]:
from models import ReviewedFinding

reviewed = [
    ReviewedFinding(
        finding=findings[0],
        verdict="confirmed",
        reviewer_notes=(
            "Verified: azurerm_application_gateway -> azapi_resource is a full "
            "resource replacement. This is expected for the AzAPI migration and "
            "is documented in UPGRADE.md. Users must plan for downtime."
        ),
    ),
    ReviewedFinding(
        finding=findings[1],
        verdict="confirmed",
        reviewer_notes=(
            "Verified: resource_group_name -> parent_id change is documented "
            "in the variable mapping table. The type change from name to ARM ID "
            "is accurate."
        ),
    ),
    ReviewedFinding(
        finding=findings[2],
        verdict="confirmed",
        reviewer_notes=(
            "Verified: map -> list shape change is consistent across all "
            "collection variables (backend pools, http settings, listeners, etc). "
            "UPGRADE.md documents each mapping."
        ),
    ),
    ReviewedFinding(
        finding=findings[3],
        verdict="confirmed",
        reviewer_notes=(
            "Verified: create_public_ip, public_ip_name, public_ip_resource_id "
            "are all removed. UPGRADE.md mentions this but it could be more "
            "prominent -- suggest adding a migration example."
        ),
    ),
]

print(f"Review results: {len(reviewed)} findings reviewed\n")
for r in reviewed:
    v = {"confirmed": "OK", "rejected": "NO", "needs_investigation": "??"}[r.verdict]
    print(f"  [{v}] {r.verdict}: {r.finding.description[:60]}...")
    print(f"       Notes: {r.reviewer_notes[:80]}...")
    print()

## 8. Report -- Generate Outputs

The Reporter agent formats the validated findings into:
- A structured test report (markdown)
- GitHub issues for confirmed critical/warning findings
- UPGRADE.md improvement suggestions

In [ ]:
from tools.reporting import (
    generate_test_report, generate_issue_body,
    generate_upgrade_doc_suggestion,
)

# Generate a test report
report_input = {
    "module_source": MODULE_PATH,
    "module_version": "feat/azapi-migration",
    "examples_tested": ["default"],
    "deploy_results": [{
        "example": "default",
        "status": "simulated",
        "plan_summary": {
            "creates": 15, "updates": 2, "deletes": 12, "replaces": 1,
        },
        "idempotency": {"status": "pass", "unexpected_changes": 0},
    }],
    "findings": [json.loads(f.to_json()) for f in findings],
}

report = json.loads(generate_test_report(json.dumps(report_input)))
print("=== Test Report ===")
print(report.get("report", "(report content)"))

In [ ]:
# Generate a GitHub issue body for the critical finding
issue = json.loads(generate_issue_body(json.dumps({
    "title": "Breaking change: Application Gateway resource replacement on AzAPI migration",
    "finding": json.loads(findings[0].to_json()),
    "module": "terraform-azurerm-avm-res-network-applicationgateway",
    "branch": "feat/azapi-migration",
})))
print("=== Issue Body ===")
print(issue.get("body", "(issue body)"))

## 9. Multi-Agent Mode

In production, the full pipeline runs as coordinated agents. Set
`MULTI_AGENT=true` to enable the hub-and-spoke topology:

```
                 Orchestrator
                /   |    |    \
         Discovery Deploy(n) Analysis
                              |
                           Reviewer
                              |
                           Reporter
```

Each agent has **focused instructions** and a **minimal tool subset**,
so context stays small and accurate. Data flows as structured JSON
(`ModuleMap`, `DeployResult`, `AnalysisFinding`, `ReviewedFinding`)
between agents -- raw terraform output never crosses agent boundaries.

### Why multi-agent?

| Problem | Single agent | Multi-agent |
|---------|-------------|-------------|
| Context fog from long TF output | Degrades analysis quality | Deploy agent discards raw output, passes summary |
| Large modules (12+ examples) | Exceeds context window | Concurrent deploy agents, one per example |
| False positives in analysis | No cross-check | Reviewer agent validates with fresh context |
| Mixed concerns | One agent does everything | Each agent is a focused specialist |

In [ ]:
# Preview multi-agent tool allocation (no Azure needed)
from agents.discovery import DISCOVERY_TOOLS
from agents.deploy import DEPLOY_TOOLS
from agents.analysis import ANALYSIS_TOOLS
from agents.reviewer import REVIEWER_TOOLS
from agents.reporter import REPORTER_TOOLS

agent_tools = {
    "Discovery": DISCOVERY_TOOLS,
    "Deploy": DEPLOY_TOOLS,
    "Analysis": ANALYSIS_TOOLS,
    "Reviewer": REVIEWER_TOOLS,
    "Reporter": REPORTER_TOOLS,
}

print("Agent tool allocation:\n")
for name, tools in agent_tools.items():
    tool_names = [t.__name__ for t in tools]
    print(f"  {name} ({len(tools)} tools): {', '.join(tool_names)}")
    print()

## 10. Cleanup

Always destroy test resources when done. The agent does this automatically
when `CLEANUP_ON_COMPLETE=true` (the default).

In [ ]:
# In production:
# terraform_destroy(workspace)
# delete_resource_group(rg_name)
# delete_workspace(workspace)

print("Cleanup would run here in a live deployment.")
print("Set CLEANUP_ON_COMPLETE=false in .env to keep resources for debugging.")

## Next Steps

1. **Run for real**: Uncomment the deploy steps in section 4 to test against Azure
2. **Test more examples**: Loop over all 12 examples, not just `default`
3. **Compare branches**: Use `clone_repo` + branch checkout to test registry versions
4. **Enable multi-agent**: Set `MULTI_AGENT=true` and run `python main.py`
5. **File issues**: Connect GitHub CLI (`gh auth login`) to auto-file bugs

See [ROADMAP.md](ROADMAP.md) for the full development plan.